[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/PerformanceEstimation/Tutorial-Europt-2026/blob/main/Part%202/Europt2026_Tutorial_Part2.ipynb)

# Second hands-on session with PEPit : gradient descent on smooth (possibly nonconvex) loss functions.

This notebook focuses on a second working example: gradient descent for minimizing smooth (possibly nonconvex) objectives. Concretely, we consider the problem

$$ f_\star=\min_{x\in\mathbb{R}^d} f(x), $$

where $f$ is continuously differentiable, has Lipschitz gradient, and is bounded below. We use gradient descent:

$$ x_{k+1} = x_k - \gamma \nabla f(x_k), $$

and study is capability to reach stationnary points. More precisely, we will study bounds of the type

$$ \min_{0\leq i\leq n} \|\nabla f(x_i)\|^2 \leq \tau_n(L,\gamma) (f(x_0)-f_\star),$$

or

$$ \min_{0\leq i\leq n} \|\nabla f(x_i)\|^2 \leq \tau_n(L,\gamma) (f(x_0)-f(x_n)),$$

and try to find the smallest possible values of $\tau_n(L,\gamma)$ so that the previous inequalities are valid.


The numerical studies below rely again on the [PEPit package](https://pepit.readthedocs.io/en/latest/).

## 1 Installing PEPit

In [ ]:
# If PEPit is not installed yet, you can run this cell.
!pip3 install pepit;

## 2 How to obtain a worst-case guarantee for gradient descent (GD) using PEPit

This section provides a step-by-step tutorial on how to use PEPit to compute a worst-case guarantee for gradient descent for $L$-smooth objectives. As before, the main focus will be on the **four main ingredients**:
- <font color='blue'>a class of problems</font> (containing the assumptions on the function to be minimized),
- <font color='red'>a first-order method</font> (to be analyzed),
- <font color='purple'>a performance measure</font> (measuring the quality of the output of the algorithm under consideration),
- <font color='green'>an initial condition</font> (measuring the quality of the initial iterate).

### 2.1 Imports
Before anything else, we have to import the PEP and the classes of functions of interest, as well as other necessary packages.

In [ ]:
# Import PEPit and the necessary class of functions
from PEPit import PEP
from PEPit.functions import SmoothFunction

# Import the CVXPY namespace for solver selection
import cvxpy as cp

# Import numpy
import numpy as np

# Import matplotlib
import matplotlib.pyplot as plt

### 2.2 Initialization of PEPit
Then, we initialize a PEP object. This object allows manipulating the forthcoming ingredients of the PEP, such as functions and iterates.

In [ ]:
# Instantiate PEP
problem = PEP()

### 2.3 Choose parameters values

For the sake of the example, we pick some simple values for the problem class and algorithmic parameters, for which we perform the worst-case analysis below.

In [ ]:
# Specify values for the smoothness parameter L, number of iterations n, and the step size gamma.
L = 1
n = 4
gamma = 1 / L

### 2.4 Specifying the problem class
Next, we specify our assumptions on the <font color='blue'>**class of functions**</font> containing the function to be optimized, and instantiate a corresponding object. Here, we consider an $L$-smooth function.

In [ ]:
# Declare an L-smooth function named "func"
func = problem.declare_function(
                SmoothFunction,
                L=L)    # Smoothness param.

We also define $x_*$ as an optimal point and $f_*$ as the optimal value, as those are needed to express our performance criteria and initial condition.

In [ ]:
# Start by defining a stationary point xs = x_* and corresponding function value fs = f_*
xs = func.stationary_point()
fs = func(xs)

### 2.5 Algorithm initialization

Third, we can instantiate the starting point for the gradient method that we will run, and specify an <font color='green'>**initial condition**</font> on this point.
To this end, a starting point $x_0$ is introduced, and a bound on the initial quality of $x_0$ is specified as $f(x_0)-f_*\leqslant 1$.

In [ ]:
# Then define the starting point x0 of the algorithm
x0 = problem.set_initial_point()

# Set the initial constraint
problem.set_initial_condition( func(x0) - fs <= 1)

### 2.6 Setting up a performance measure

It is crucial for the worst-case analysis to specify the <font color='purple'>**performance metric**</font> for which we wish to compute a worst-case performance. In this example, we wish to compute the worst-case value of $\min_{0\leq i\leq n}\|\nabla f(x_i)\|^2$. By default, the way to study the **best** iterates in PEPit is to specify **several** performance metric (which will be put in epigraph form).

We start by specifying that at least $\|\nabla f(x_0)\|^2$ is a performance metric. We will then specify the ones for the other iterates when defining the algorithm update.

In [ ]:
# Set the performance metric to include \|\nabla f(x_0)\|^2
problem.set_performance_metric( func.gradient(x0)**2 )

### 2.7 Algorithm implementation

In this fourth step, we specify the <font color='red'>**algorithm**</font> in a natural format. In this example, we simply use the iterates (which are PEPit objects) as if we had to implement gradient descent in practice using a simple loop. Gradient descent with fixed step size $\gamma$ may be described as follows, for $k \in \{0,1, \ldots, n-1\}$
\begin{equation}
x_{k+1} = x_k - \gamma \nabla f(x_k).
\end{equation}
On top of this, for each iteration, we specify and additional element to the set of performance metrics.

In [ ]:
# Run n steps of gradient descent with step-size gamma
x = x0
for _ in range(n):
    x = x - gamma * func.gradient(x)
    problem.set_performance_metric( func.gradient(x)**2 )

### 2.8 Solving the PEP

The last natural stage in the process is to solve the corresponding PEP. This is done via the following line, which will ask PEPit to perform the modelling steps and to call an appropriate SDP solver (which should be installed beforehand) to perform the worst-case analysis.

Solving the SDP is handled by the [CVXPY](https://www.cvxpy.org/) modelling layer.

The default CVXPY solver [SCS](https://web.stanford.edu/~boyd/papers/scs.html)  may not be the most appropriate  for solving SDPs; we advise using [CLARABEL](https://clarabel.org/) (free, open-source, available in colab) or [MOSEK](https://www.mosek.com/) (commercial, free academic license) if possible, for improved numerical performances.

In [ ]:
# Solve the PEP with verbose = 1 and CLARABEL solver
pepit_tau = problem.solve(verbose=1, solver=cp.CLARABEL)
print(f"\n\n*** Exact worst-case convergence bound of GD for {L}-smooth function after {n} steps is {pepit_tau}")

Woops... the problem appear to be unbounded! Can you identify an issue?

> your turn!

Hint 1: why would this ever be bounded? Can we construct examples showing that the solution of this problem has no reason to be bounded?

Hint 2: What about applying GD on $-\frac{L}{2}x^2$? Does it meet our specifications? What is wrong?

### 2.9 Fixing the PEP

An issue is that, since $f$ is not assumed to be convex, there is currently no relationship between $f_*$ and the iterates. One way to see it is simply that we do not require the problem to be lower-bounded. Another way to create problems even if $x_*$ was actually a local minimizer, is that $x_*$ could just be "*in another valley*" unrelated to the iterates.

For solving this, we need to impose a relationship between the $x_*$ under consideration and the iterates we study. One possibility is to require that all the $f(x_i)$ are actually *upper bounds* on $f(x_*)$ (which would, e.g., be valid if $f(x_*)$ was the global optimum). This corresponds to study *any $x_*$ so that $f(x_*)$ is a lower-bound on the observed function values*.

The following code puts everything together with those additional requirements:

In [ ]:
# Instantiate PEP
problem = PEP()

# Declare an L-smooth function named "func"
func = problem.declare_function(
                SmoothFunction,
                L=L)    # Smoothness param.

# Specify values for the smoothness parameter L, number of iterations n, and the step size gamma.
L = 1
n = 2
gamma = 1 / L

# Start by defining a stationary point xs = x_* and corresponding function value fs = f_*
xs = func.stationary_point()
fs = func(xs)

# Then define the starting point x0 of the algorithm
x0 = problem.set_initial_point()

# Set the initial constraint
problem.set_initial_condition( func(x0) - fs <= 1)

# Set the performance metric to include \|\nabla f(x_0)\|^2
problem.set_performance_metric( func.gradient(x0)**2 )
# f(x_0) >= f(x*)
problem.add_constraint( func(x0) >= fs )

# Run n steps of gradient descent with step-size gamma
x = x0
for _ in range(n):
    x = x - gamma * func.gradient(x)
    problem.set_performance_metric( func.gradient(x)**2 )
    # f(x) >= f(x*)
    problem.add_constraint( func(x) >= fs )


# Solve the PEP with verbose = 1 and CLARABEL solver
pepit_tau = problem.solve(verbose=1, solver=cp.CLARABEL)
print(f"\n\n*** Exact worst-case convergence bound of GD for {L}-smooth function after {n} steps is {pepit_tau}")

### 2.10 Create a Python routine and plotting worst-case guarantees as functions of number of steps

Complete the code below for a more convenient evaluation of PEP for different values of the parameters (stepsize and number of iterations).

In [ ]:
def wc_gradient_descent(L, gamma, n, verbose=1):
 
    # Instantiate PEP
    problem = PEP()

    # Declare a smooth strongly convex function
    func = problem.declare_function(SmoothFunction, L=L)
    
    # Start by defining a stationary point xs = x_* and corresponding function value fs = f_*
    xs = func.stationary_point()
    fs = func(xs)

    # Then define the starting point x0 of the algorithm as well as corresponding gradient and function value g0 and f0
    x0 = problem.set_initial_point()
    g0, f0 = func.oracle(x0)

    # Initialize the running 'x' and gradient to run GD
    x = x0
    gx, fx = g0, f0

    # Set the performance metric to the minimum of the gradient norm over the iterations
    problem.set_performance_metric(gx ** 2)
    problem.add_constraint( func(x) >= fs )

    # Run n steps of GD method with step-size gamma
    for i in range(n):
        x = x - gamma * gx
        # Set the performance metric to the minimum of the gradient norm over the iterations
        gx, fx = func.oracle(x)
        problem.set_performance_metric(gx ** 2)
        problem.add_constraint( func(x) >= fs )

    # Set the initial constraint that is the difference between f0 and fs
    problem.set_initial_condition(f0 - fs <= 1)

    # Solve the PEP
    pepit_verbose = max(verbose, 0)
    pepit_tau = problem.solve(verbose=pepit_verbose, solver=cp.CLARABEL)
    if gamma == 1/L: 
        guessed_tau = 4/3/n
    else:
        guessed_tau = None

    return pepit_tau, guessed_tau

In [ ]:
# Set the parameters
L = 1        # smoothness parameter
gamma = 1/L  # stepsize
n_max = 25   # max number of iterations to be studied

# Set a list of numner of steps
ns = range(1,n_max)

# Compute numerical and theoretical (analytical) worst-case guarantees for the each step-size
pepit_taus = list()
guessed_taus = list()
for n in ns:
    pepit_tau, guessed_tau = wc_gradient_descent( L=L, gamma=gamma, n=n, verbose=0 )
    pepit_taus.append(pepit_tau)
    guessed_taus.append(guessed_tau)
    print(f'{n+1} / {n_max} grid points computed', end='\r', flush=True)

In [ ]:
# Plot theoretical and PEPit (numerical) worst-case performance bounds as functions of the number of steps
plt.plot(ns, guessed_taus, '--', label='Guessed tight bound')
plt.plot(ns, pepit_taus, 'x', label='PEPit worst-case bound')

plt.legend()
plt.xlabel('Number of steps ($n$)')
plt.ylabel('Worst-case guarantee')

plt.show()

How could you have guessed the values of the worst-case guarantees as a function of $n$? 
> your turn!

Hint: plot the inverse of the taus! Do you recover the "guessed_tau" from previous cells?

In [ ]:
inverse_pepit_taus = [1/tau for tau in pepit_taus]
plt.plot(ns, inverse_pepit_taus, 'x', label='Inverse PEPit worst-case bound')

plt.legend()
plt.xlabel('Number of steps ($n$)')
plt.ylabel('Inverse worst-case guarantee')

plt.show()

### 2.11 Refine the analysis

One can remark that, in fact, the lower bound property $f(x_*)\leq f(x_i)$ is not very accurate, and one can be a bit more precise.

In particular, we know that for all $x\in\mathbb{R}^d$, due to smoothness:
$$ f(x)\leq f(x_i) + \langle \nabla f(x_i),x-x_i\rangle + \tfrac L 2 \|x-x_i\|^2. $$

Hence, minimizing this bound, one can get:

$$ f_*\leq f(x_i-\tfrac1L \nabla f(x_i))\leq f(x_i) - \tfrac 1 {2L} \|\nabla f(x_i)\|^2.$$

This can be reflected in our code as shown below.

In [ ]:
def wc_gradient_descent_refined(L, gamma, n, verbose=1):
 
    # Instantiate PEP
    problem = PEP()

    # Declare a smooth strongly convex function
    func = problem.declare_function(SmoothFunction, L=L)
    
    # Start by defining a stationary point xs = x_* and corresponding function value fs = f_*
    xs = func.stationary_point()
    fs = func(xs)

    # Then define the starting point x0 of the algorithm as well as corresponding gradient and function value g0 and f0
    x0 = problem.set_initial_point()
    g0, f0 = func.oracle(x0)

    # Initialize the running 'x' and gradient to run GD
    x = x0
    gx, fx = g0, f0

    # Set the performance metric to the minimum of the gradient norm over the iterations
    problem.set_performance_metric(gx ** 2)
    problem.add_constraint( func(x)- 1/2/L * gx**2 >= fs )

    # Run n steps of GD method with step-size gamma
    for i in range(n):
        x = x - gamma * gx
        # Set the performance metric to the minimum of the gradient norm over the iterations
        gx, fx = func.oracle(x)
        problem.set_performance_metric(gx ** 2)
        problem.add_constraint( func(x) - 1/2/L * gx**2 >= fs )

    # Set the initial constraint that is the difference between f0 and fs
    problem.set_initial_condition(f0 - fs <= 1)

    # Solve the PEP
    pepit_verbose = max(verbose, 0)
    pepit_tau = problem.solve(verbose=pepit_verbose, solver=cp.CLARABEL)
    if gamma == 1/L:
        guessed_tau = 4 / ( 3 * n + 2 ) # YOUR GUESS HERE
    else:
        guessed_tau = None

    return pepit_tau, guessed_tau

In [ ]:
# Set the parameters
L = 1        # smoothness parameter
gamma = 1/L  # stepsize
n_max = 25   # max number of iterations to be studied

# Set a list of numner of steps
ns = range(1,n_max)

# Compute numerical and theoretical (analytical) worst-case guarantees for the each step-size
pepit_taus = list()
guessed_taus = list()
for n in ns:
    pepit_tau, guessed_tau = wc_gradient_descent_refined( L=L, gamma=gamma, n=n, verbose=0 )
    pepit_taus.append(pepit_tau)
    guessed_taus.append(guessed_tau)
    print(f'{n+1} / {n_max} grid points computed', end='\r', flush=True)

In [ ]:
# Plot theoretical and PEPit (numerical) worst-case performance bounds as functions of the number of steps
plt.plot(ns, guessed_taus, '--', label='Guessed tight bound')
plt.plot(ns, pepit_taus, 'x', label='PEPit worst-case bound')

plt.legend()
plt.xlabel('Number of steps ($n$)')
plt.ylabel('Worst-case guarantee')

plt.show()

How could you have guessed the values of the worst-case guarantees as a function of $n$ again? 
> your turn!

Hint: same answer :)

### 2.12 Optimal stepsize? Plotting worst-case guarantees as functions of the step size

In [ ]:

# Set the parameters
n = 4      # iteration counter
L = 1      # smoothness parameter
nb_grid_elem = 50 # size of the grid of stepsizes

# Set a list of step-sizes to test
gammas = np.linspace(.1/L, 1.9 / L, nb_grid_elem)

# Compute numerical and theoretical (analytical) worst-case guarantees for the each step-size
pepit_taus = list()
for i in range(nb_grid_elem):
    gamma = gammas[i]
    pepit_tau, _ = wc_gradient_descent_refined( L=L, gamma=gamma, n=n, verbose=0 )
    pepit_taus.append(pepit_tau)
    print(f'{i+1} / {nb_grid_elem} grid points computed', end='\r', flush=True)

In [ ]:
my_guessed_optimal_stepsize = 2/L/np.sqrt(3)  # <-- replace with your value


# Compute numerically optimal stepsize
idx_min = np.argmin(pepit_taus)
optimal_stepsize = gammas[idx_min]
minimal_tau = pepit_taus[idx_min]
print(f'Optimal (numerically identified) stepsize is: {optimal_stepsize:.4g}')
print(f'Guessed optimal stepsize is: {my_guessed_optimal_stepsize:.4g}')

# Plots
plt.plot(gammas, pepit_taus, 'x', label='PEPit worst-case bound')
plt.axvline(
    x=my_guessed_optimal_stepsize,
    linestyle=':',
    color='orange',
    linewidth=2,
    label='Guessed optimal stepsize'
)
plt.legend(
    loc='upper left',
    bbox_to_anchor=(1.02, 1),
    borderaxespad=0
)

plt.xlabel(r'Step-size ($\gamma$)')
plt.ylabel('Worst-case guarantee')

plt.tight_layout()
plt.show()


## 3 Alternate performance criteria and initial conditions

### 3.1 Change the performance criteria

Edit the previous code to study the criterion $\sum_{i=0}^n \|\nabla f(x_i)\|^2$ and report the differences.

In [ ]:
def wc_gradient_descent_sum_gradients(L, gamma, n, verbose=1):
 
    # Instantiate PEP
    problem = PEP()

    # Declare a smooth strongly convex function
    func = problem.declare_function(SmoothFunction, L=L)
    
    # Start by defining a stationary point xs = x_* and corresponding function value fs = f_*
    xs = func.stationary_point()
    fs = func(xs)

    # Then define the starting point x0 of the algorithm as well as corresponding gradient and function value g0 and f0
    x0 = problem.set_initial_point()
    g0, f0 = func.oracle(x0)

    # Initialize the running 'x' and gradient to run GD
    x = x0
    gx, fx = g0, f0

    # Initialize the sum of gradients
    sum_gradients = gx ** 2
    problem.add_constraint( func(x)- 1/2/L * gx**2 >= fs )

    # Run n steps of GD method with step-size gamma
    for i in range(n):
        x = x - gamma * gx
        # Set the performance metric to the minimum of the gradient norm over the iterations
        gx, fx = func.oracle(x)
        sum_gradients = sum_gradients + gx ** 2
        problem.add_constraint( func(x)- 1/2/L * gx**2 >= fs )

    # Set the performance metric to sum of gradients
    problem.set_performance_metric(sum_gradients)
    # Set the initial constraint that is the difference between f0 and fs
    problem.set_initial_condition(f0 - fs <= 1)

    # Solve the PEP
    pepit_verbose = max(verbose, 0)
    pepit_tau = problem.solve(verbose=pepit_verbose, solver=cp.CLARABEL)
    guessed_tau = 0 # COMPLETE YOUR GUESS HERE

    return pepit_tau, guessed_tau

In [ ]:
# Set the parameters
L = 1        # smoothness parameter
gamma = 1/L  # stepsize
n_max = 25   # max number of iterations to be studied

# Set a list of numner of steps
ns = range(1,n_max)

# Compute numerical and theoretical (analytical) worst-case guarantees for the each step-size
pepit_taus = list()
guessed_taus = list()
for n in ns:
    pepit_tau, guessed_tau = wc_gradient_descent_sum_gradients( L=L, gamma=gamma, n=n, verbose=0 )
    pepit_taus.append(pepit_tau)
    guessed_taus.append(guessed_tau)
    print(f'{n+1} / {n_max} grid points computed', end='\r', flush=True)

In [ ]:
# Plot theoretical and PEPit (numerical) worst-case performance bounds as functions of the number of steps
plt.plot(ns, guessed_taus, '--', label='Guessed tight bound')
plt.plot(ns, pepit_taus, 'x', label='PEPit worst-case bound')

plt.legend()
plt.xlabel('Number of steps ($n$)')
plt.ylabel('Worst-case guarantee')

plt.show()

Using the following bound $$(n+1) \min_{0\leq i \leq n} \|\nabla f(x_i)\|^2 \leq \sum_{i=0}^{n} \|\nabla f(x_i)\|^2.$$

What do you conclude on the best gradient norm? Does this bound match your previous estimate? Are the rates and constant matching?

### 3.2 Change the initial condition

Go back to the performance criterion being $\min_{0\leq i \leq n} \|\nabla f(x_i)\|^2 $ and edit previous codes to substitute the condition $f(x_0)-f_*\leq 1$ by $f(x_0)-f(x_n)\leq 1$.

In [ ]:
def wc_gradient_descent_no_fs(L, gamma, n, verbose=1):
 
    # Instantiate PEP
    problem = PEP()

    # Declare a smooth strongly convex function
    func = problem.declare_function(SmoothFunction, L=L)
    
    # Start by defining a stationary point xs = x_* and corresponding function value fs = f_*
    xs = func.stationary_point()
    fs = func(xs)

    # Then define the starting point x0 of the algorithm as well as corresponding gradient and function value g0 and f0
    x0 = problem.set_initial_point()
    g0, f0 = func.oracle(x0)

    # Initialize the running 'x' and gradient to run GD
    x = x0
    gx, fx = g0, f0

    # Set the performance metric to the minimum of the gradient norm over the iterations
    problem.set_performance_metric(gx ** 2)

    # Run n steps of GD method with step-size gamma
    for i in range(n):
        x = x - gamma * gx
        # Set the performance metric to the minimum of the gradient norm over the iterations
        gx, fx = func.oracle(x)
        problem.set_performance_metric(gx ** 2)

    # Set the initial constraint that is the difference between f0 and fs
    problem.set_initial_condition(f0 - fx <= 1)

    # Solve the PEP
    pepit_verbose = max(verbose, 0)
    pepit_tau = problem.solve(verbose=pepit_verbose, solver=cp.CLARABEL)
    guessed_tau = 0 # YOUR GUESS HERE

    return pepit_tau, guessed_tau

In [ ]:
# Set the parameters
L = 1        # smoothness parameter
gamma = 1/L  # stepsize
n_max = 25   # max number of iterations to be studied

# Set a list of numner of steps
ns = range(1,n_max)

# Compute numerical and theoretical (analytical) worst-case guarantees for the each step-size
pepit_taus = list()
guessed_taus = list()
for n in ns:
    pepit_tau, guessed_tau = wc_gradient_descent_no_fs( L=L, gamma=gamma, n=n, verbose=0 )
    pepit_taus.append(pepit_tau)
    guessed_taus.append(guessed_tau)
    print(f'{n+1} / {n_max} grid points computed', end='\r', flush=True)

In [ ]:
# Plot theoretical and PEPit (numerical) worst-case performance bounds as functions of the number of steps
plt.plot(ns, guessed_taus, '--', label='Guessed tight bound')
plt.plot(ns, pepit_taus, 'x', label='PEPit worst-case bound')

plt.legend()
plt.xlabel('Number of steps ($n$)')
plt.ylabel('Worst-case guarantee')

plt.show()

### Going further <a class="anchor" id="sec4"></a>

Interpolation conditions for smooth functions can be found in [2], while complete analyses of gradient descent and optimal stepsizes for this setting can be found in [3] and [4].

More complete references and demos can be found in [PEPit's documentation and demos](https://github.com/PerformanceEstimation/PEPit/blob/master/ressources/demo).


---

### References

[1] Y. Drori and M. Teboulle, *Performance of first-order methods for smooth convex minimization: a novel approach*, Mathematical Programming, 2014. [arXiv:1206.3209](https://arxiv.org/abs/1206.3209)

[2] A. B. Taylor, J. M. Hendrickx, and F. Glineur, *Exact worst-case performance of first-order methods for composite convex optimization*, SIAM Journal on Optimization, 2017. [arXiv:1512.07516](https://arxiv.org/pdf/1512.07516)

[3] H. Abbaszadehpeivasti, E. de Klerk, M. Zamani. *The exact worst-case convergence rate of the gradient method with fixed step lengths for L-smooth functions*, Optimization Letters, 2022. [https://doi.org/10.1007/s11590-021-01821-1](https://doi.org/10.1007/s11590-021-01821-1)

[4] T. Rotaru, F. Glineur, P. Patrinos. *Exact worst-case convergence rates of gradient descent: a complete analysis for all constant stepsizes over nonconvex and convex functions*, Mathematical Programming, 2022. [arXiv:2406.17506](https://arxiv.org/pdf/2406.17506)